
# Banking Stock Data Loading into PostgreSQL

## Phase 5 — Database Integration

### Business Objective
Load validated Indian banking stock observations into PostgreSQL for financial market intelligence and Power BI reporting.

### Source Dataset
Data/processed/banking_stocks_processed.csv

### Target Database
financial_market_analytics

### Target Table
public.banking_stocks

### Expected Records
- HDFCBANK.BSE: 99
- ICICIBANK.BSE: 99
- SBIN.BSE: 99
- Total: 297

### Loading Strategy
1. Read the validated stock dataset.
2. Verify record counts, schema, and missing values.
3. Verify primary-key uniqueness.
4. Connect securely to PostgreSQL.
5. Insert data into the existing table.
6. Reconcile the loaded records.

### Data Integrity
- Preserve the original raw dataset.
- Retain all 21 statistical outliers.
- Do not recreate or replace the existing PostgreSQL table.
- Preserve all database constraints.


In [1]:

from pathlib import Path
import sys

# Locate the project root.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]
elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Support existing notebook variables.
PROJECT_ROOT = project_root

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("Banking_Stocks_PostgreSQL_Loading")

logger.info(
    "Banking stocks PostgreSQL loading notebook initialized"
)


2026-11-02 16:16:53,848 | INFO | Banking_Stocks_PostgreSQL_Loading | Banking stocks PostgreSQL loading notebook initialized


In [2]:

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Load database credentials.
load_dotenv(PROJECT_ROOT / ".env")

# Create a separate connection for read-only verification.
verification_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

verification_engine = create_engine(
    verification_url,
    pool_pre_ping=True
)

# Verify existing database records.
with verification_engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT current_database()")
    ).scalar_one()

    stock_count = connection.execute(
        text("SELECT COUNT(*) FROM public.banking_stocks")
    ).scalar_one()

logger.info("Banking stocks PostgreSQL verification started")
logger.info("Connected database: %s", database_name)
logger.info("Database table: public.banking_stocks")
logger.info("Existing banking stock records: %s", stock_count)

if (
    database_name == "financial_market_analytics"
    and stock_count == 297
):
    logger.info("Banking stocks PostgreSQL verification PASSED")
else:
    logger.error("Banking stocks PostgreSQL verification FAILED")


2026-11-02 16:19:08,233 | INFO | Banking_Stocks_PostgreSQL_Loading | Banking stocks PostgreSQL verification started
2026-11-02 16:19:08,244 | INFO | Banking_Stocks_PostgreSQL_Loading | Connected database: financial_market_analytics
2026-11-02 16:19:08,246 | INFO | Banking_Stocks_PostgreSQL_Loading | Database table: public.banking_stocks
2026-11-02 16:19:08,247 | INFO | Banking_Stocks_PostgreSQL_Loading | Existing banking stock records: 297
2026-11-02 16:19:08,249 | INFO | Banking_Stocks_PostgreSQL_Loading | Banking stocks PostgreSQL verification PASSED


In [1]:

from pathlib import Path
import pandas as pd

# Locate project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

# Processed stock dataset path
STOCK_FILE = (
    PROJECT_ROOT / "Data" / "processed"
    / "banking_stocks_processed.csv"
)

assert STOCK_FILE.exists(), (
    f"Processed stock dataset not found: {STOCK_FILE}"
)

# Load validated stock dataset
df_stock_load = pd.read_csv(
    STOCK_FILE,
    parse_dates=["observation_date"]
)

# Expected schema
EXPECTED_COLUMNS = [
    "observation_date",
    "stock_symbol",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "source",
    "price_outlier",
    "volume_outlier",
    "is_outlier"
]

# Validate schema and row count
assert df_stock_load.columns.tolist() == EXPECTED_COLUMNS
assert len(df_stock_load) == 297

# Validate missing values
assert df_stock_load.isna().sum().sum() == 0

# Validate primary-key uniqueness
assert not df_stock_load.duplicated(
    subset=["observation_date", "stock_symbol"]
).any()

# Validate outlier counts
assert int(df_stock_load["price_outlier"].sum()) == 0
assert int(df_stock_load["volume_outlier"].sum()) == 21
assert int(df_stock_load["is_outlier"].sum()) == 21

# Validate stock distribution
expected_counts = {
    "HDFCBANK.BSE": 99,
    "ICICIBANK.BSE": 99,
    "SBIN.BSE": 99
}

assert (
    df_stock_load["stock_symbol"]
    .value_counts()
    .to_dict()
    == expected_counts
)

print("Dataset Shape:", df_stock_load.shape)

print("\nStock Distribution:")
print(df_stock_load["stock_symbol"].value_counts())

print("\nMissing Values:")
print(df_stock_load.isna().sum())

print("\nStatistical Outliers:")
print("Price Outliers:", df_stock_load["price_outlier"].sum())
print("Volume Outliers:", df_stock_load["volume_outlier"].sum())
print("Total Flagged Records:", df_stock_load["is_outlier"].sum())

print("\nDataset Preview:")
display(df_stock_load.head())

print("\nBanking Stock Pre-Loading Validation: PASSED")


Dataset Shape: (297, 11)

Stock Distribution:
stock_symbol
HDFCBANK.BSE     99
ICICIBANK.BSE    99
SBIN.BSE         99
Name: count, dtype: int64

Missing Values:
observation_date    0
stock_symbol        0
open_price          0
high_price          0
low_price           0
close_price         0
volume              0
source              0
price_outlier       0
volume_outlier      0
is_outlier          0
dtype: int64

Statistical Outliers:
Price Outliers: 0
Volume Outliers: 21
Total Flagged Records: 21

Dataset Preview:


,observation_date,stock_symbol,open_price,high_price,low_price,close_price,volume,source,price_outlier,volume_outlier,is_outlier
0,2026-05-22,HDFCBANK.BSE,759.00,774.90,759.00,766.40,629420,Alpha Vantage,False,False,False
1,2026-05-25,HDFCBANK.BSE,778.00,788.00,775.25,786.85,964192,Alpha Vantage,False,False,False
2,2026-05-26,HDFCBANK.BSE,785.95,790.90,777.00,779.00,932064,Alpha Vantage,False,False,False
3,2026-05-27,HDFCBANK.BSE,772.10,772.10,756.60,758.50,4893862,Alpha Vantage,False,False,False
4,2026-05-29,HDFCBANK.BSE,749.00,760.65,738.10,744.75,1799471,Alpha Vantage,False,False,False



Banking Stock Pre-Loading Validation: PASSED



## 2. Secure PostgreSQL Connection and Banking Stock Data Loading

### Business Objective
Load validated banking stock observations into the financial_market_analytics PostgreSQL database.

### Source
Data/processed/banking_stocks_processed.csv

### Target
public.banking_stocks

### Loading Method
1. Establish a secure PostgreSQL connection using .env.
2. Verify the target database and table.
3. Confirm that the target table is empty.
4. Insert 297 validated observations.
5. Preserve existing primary keys and CHECK constraints.
6. Verify the inserted record count.

### Data Integrity
- Use a transaction to prevent partial insertion.
- Do not recreate or replace the table.
- Prevent accidental duplicate loading.
- Preserve the 21 flagged trading-volume observations.

In [2]:

import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Load secure database configuration
ENV_PATH = PROJECT_ROOT / ".env"

assert ENV_PATH.exists(), ".env file not found."

load_dotenv(ENV_PATH)

required_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

missing_variables = [
    key for key in required_variables
    if not os.getenv(key)
]

assert not missing_variables, (
    f"Missing environment variables: {missing_variables}"
)

# Create PostgreSQL connection
database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

# Verify database and target table
with engine.connect() as connection:

    database_name = connection.execute(
        text("SELECT current_database()")
    ).scalar_one()

    table_exists = connection.execute(
        text("""
            SELECT EXISTS (
                SELECT 1
                FROM information_schema.tables
                WHERE table_schema = 'public'
                  AND table_name = 'banking_stocks'
            )
        """)
    ).scalar_one()

    existing_records = connection.execute(
        text(
            "SELECT COUNT(*) FROM public.banking_stocks"
        )
    ).scalar_one()

print("PostgreSQL Connection: SUCCESS")
print("Connected Database:", database_name)
print("Banking Stocks Table Exists:", table_exists)
print("Existing Banking Stock Records:", existing_records)

assert database_name == "financial_market_analytics"
assert table_exists
assert existing_records == 0, (
    "Target table is not empty. "
    "Loading stopped to prevent duplicate insertion."
)

# Insert records within a single transaction
with engine.begin() as connection:

    inserted_rows = df_stock_load.to_sql(
        name="banking_stocks",
        con=connection,
        schema="public",
        if_exists="append",
        index=False,
        chunksize=100,
        method="multi"
    )

print("\nBanking Stock Data Insertion: COMPLETED")
print("Rows Reported Inserted:", inserted_rows)

# Verify final record count
with engine.connect() as connection:

    loaded_count = connection.execute(
        text(
            "SELECT COUNT(*) FROM public.banking_stocks"
        )
    ).scalar_one()

assert loaded_count == len(df_stock_load), (
    "Database row count does not match source dataset."
)

print("Source Records:", len(df_stock_load))
print("Database Records:", loaded_count)
print("Record Count Match:", loaded_count == len(df_stock_load))

print("\nBanking Stock PostgreSQL Data Loading: PASSED")


PostgreSQL Connection: SUCCESS
Connected Database: financial_market_analytics
Banking Stocks Table Exists: True
Existing Banking Stock Records: 0

Banking Stock Data Insertion: COMPLETED
Rows Reported Inserted: 297
Source Records: 297
Database Records: 297
Record Count Match: True

Banking Stock PostgreSQL Data Loading: PASSED
